[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/landscape_transience/common_analyses_to_check_for_transience.ipynb)

# Common analyses that you can use to check for landscape transience

*This lesson made by Simon M Mudd. It is the `lsdtt3` version of an earlier `lsdtt2` notebook.*

A landscape is **transient** when it is still adjusting to a change, for example a change in uplift rate or base level. Transient landscapes often have a steep, rapidly eroding lower part and a gentler, slowly eroding upper part, separated by **knickpoints** in the river profiles. This notebook runs a few standard analyses that help you spot this. Our example is the Aspromonte and the Serre mountains of Calabria, southern Italy, which are being uplifted rapidly.

In this notebook you will:

1. Download a 30 m digital elevation model (DEM) with `lsdtt-fetch-raster`.
2. Make a hillshade, and maps of topographic gradient and curvature, with `lsdtt-surface-metrics`.
3. Plot a histogram of topographic gradient.
4. Extract basins from outlet points and run a **chi analysis** with `lsdtt-chi-analysis`.
5. Plot chi profiles and channel long profiles.
6. Make a **swath profile** with `lsdtt-swath-profile`.
7. Map the channel steepness index $k_{sn}$.

The `lsdtt3` programs are command-line tools controlled by small text **parameter files** (`key: value` lines). We write these files from Python and then run the programs with `!`.

## READ THIS FIRST

**To use your own landscape you only need to change a few things.** Search for the text **"Edit code block below for your site"** to find them:

* The box around your study area (Step 1).
* The basin outlets (Step 4).
* The basin to plot (Step 5).
* The two ends of the swath (Step 6).

Every figure is saved as a png, with a file name that tells you what it is.

## Setup

### First set up condacolab

This step takes around 2 minutes. **`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now install `pygmt` (GMT is the Generic Mapping Tools). This is the slowest step and takes around a minute.

In [ ]:
!mamba install pygmt

Now get `lsdviztools3`, the plotting package.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.1.0-py3-none-any.whl

In [ ]:
!pip install "lsdviztools3-0.1.0-py3-none-any.whl[charts]"

Next, download and unpack the `lsdtt3` command-line programs.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

Tell the system where to find the `lsdtt3` programs and their libraries. If your runtime restarts later, run this cell again.

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.5.2"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's older proj.db breaks rasterio and pyproj.

Check that it works: this prints the version of `lsdtt-chi-analysis`.

In [ ]:
!lsdtt-chi-analysis -v

## Step 1: Download a DEM

`lsdtt-fetch-raster` downloads a DEM for a box given in longitude and latitude (WGS84 degrees):

* `dem_source: cop30_aws`: the Copernicus GLO-30 DEM (about 30 m resolution), served from AWS. No account or API key is needed (the `lsdtt2` version of this notebook needed an OpenTopography API key).
* `target_epsg: 32633`: reproject to **UTM zone 33N**, a coordinate system in metres. Gradient, curvature, flow routing and chi all need distances in metres, not degrees. If you change the site, use the UTM zone of your area: zone number = floor((longitude + 180) / 6) + 1, with EPSG code 326xx in the northern hemisphere and 327xx in the southern hemisphere. Or delete this line and the tool will choose the zone for you.
* `grid_spacing: 30`: a 30 m output grid.

The output is called `<write_fname>_DEM.tif`. We put the prefix in a variable, `prefix`, which every later step uses.

### Edit code block below for your site!

* Change `west`, `east`, `south` and `north` (decimal degrees). You can get a latitude and longitude by right clicking on Google Maps.
* Change `target_epsg` to the UTM zone of your site (see above).
* You can change `prefix` too: all the output files start with it.

In [ ]:
prefix = "calabria"

west, east = 15.628317625157045, 16.210748067627133
south, north = 37.87521641344407, 38.30554438686832
target_epsg = 32633

# Write the parameter file for lsdtt-fetch-raster
with open(f"{prefix}_fetch.param", "w") as f:
    f.write(f"write_fname: {prefix}\n")
    f.write(f"west: {west}\n")
    f.write(f"east: {east}\n")
    f.write(f"south: {south}\n")
    f.write(f"north: {north}\n")
    f.write("dem_source: cop30_aws\n")
    f.write(f"target_epsg: {target_epsg}\n")
    f.write("grid_spacing: 30\n")

# Run it. The first argument is the directory holding the parameter file.
!lsdtt-fetch-raster ./ {prefix}_fetch.param

**Important**: you should always cite your data source. For this data, you should cite:

*European Space Agency, Sinergise (2021). Copernicus Global Digital Elevation Model. Distributed by OpenTopography. https://doi.org/10.5069/G9028PQB*

## Step 2: Hillshade, gradient and curvature

Topographic gradient and hillslope curvature give some indication of how quickly a landscape is eroding. In a transient landscape you often find steeper slopes, and sharper hilltops, in the part that has "felt" the change.

`lsdtt-surface-metrics` computes them by fitting a smooth polynomial surface to the elevations around each pixel and taking the derivatives of that surface. The method follows:

*Hurst, M.D., Mudd, S.M., Walcott, R., Attal, M., Yoo, K., 2012. Using hilltop curvature to derive the spatial distribution of erosion rates. J. Geophys. Res. 117, F02017. https://doi.org/10.1029/2011JF002057*

The parameters:

* `write_hillshade_raster: true`: a hillshade lit from the northwest (`hillshade_azimuth: 315`) at 45 degrees above the horizon (`hillshade_altitude: 45`). Output: `calabria_hillshade.tif`.
* `polyfit_lambda_metres: 70`: the size of the window used in the fit. 70 m suits 30 m data. (In `lsdtt2` this was `surface_fitting_radius`.)
* `polyfit_metrics: slope,laplacian`: the topographic gradient (m/m) and the curvature (1/m). The `laplacian` is the sum of the second derivatives of the surface, which is the curvature that `lsdtt2` called `CURV`. **Hilltops (convex) have negative curvature, valleys (concave) have positive curvature.** Note: there must be no space after the comma. Outputs: `calabria_direct_polyfit_slope.tif` and `calabria_direct_polyfit_laplacian.tif`.

In [ ]:
with open(f"{prefix}_metrics.param", "w") as f:
    f.write(f"read_fname: {prefix}_DEM.tif\n")
    f.write(f"write_fname: {prefix}\n")
    f.write("write_hillshade_raster: true\n")
    f.write("hillshade_azimuth: 315\n")
    f.write("hillshade_altitude: 45\n")
    f.write("polyfit_lambda_metres: 70\n")
    f.write("polyfit_metrics: slope,laplacian\n")

!lsdtt-surface-metrics ./ {prefix}_metrics.param

In [ ]:
!ls -lh {prefix}_*

### Figure 1: Shaded relief

`render_hillshade` with `shade=True` colours the DEM by elevation and adds hillshade illumination.

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from IPython.display import Image, display

from lsdviztools3.io.vector import load_vector
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.channels import render_drape, render_channels
from lsdviztools3.render.basins import render_basins
from lsdviztools3.render.points import render_points
from lsdviztools3.render.base import save_figure

style = MapStyle(figure_size="15c", transparent=False)
hs_style = replace(style, cmap="gray")

fig = render_hillshade(f"{prefix}_DEM.tif", shade=True, style=replace(style, cmap="gist_earth"))
save_figure(fig, f"{prefix}_relief_plot.png", style=style)
display(Image(f"{prefix}_relief_plot.png", width=800))

### Figure 2: Topographic gradient

`render_drape` draws the hillshade in gray and lays the gradient on top in colour. `transparency` sets how much of the hillshade shows through (0 = solid colour, 100 = invisible). We clip the colour scale at the 2nd and 98th percentiles (`percentile_min`, `percentile_max`) so a few extreme pixels do not wash out the map.

In [ ]:
fig = render_drape(f"{prefix}_hillshade.tif", f"{prefix}_direct_polyfit_slope.tif",
                   base_is_hillshade=True, transparency=30, cmap="inferno",
                   percentile_min=2, percentile_max=98,
                   colorbar=True, colorbar_label="Gradient (m/m)", colorbar_position="right",
                   style=style)
save_figure(fig, f"{prefix}_gradient_plot.png", style=style)
display(Image(f"{prefix}_gradient_plot.png", width=800))

### Figure 3: Curvature

Curvature is positive and negative, so we use a diverging colour map (`RdBu`) and make the colour scale symmetric around zero: red is convex (hilltops and ridges), blue is concave (valleys). The scale runs to the 98th percentile of the absolute curvature.

In [ ]:
with rasterio.open(f"{prefix}_direct_polyfit_laplacian.tif") as src:
    curv = src.read(1, masked=True).compressed()
curv_lim = float(np.percentile(np.abs(curv[np.isfinite(curv)]), 98))
print(f"Colour scale: -{curv_lim:.4f} to {curv_lim:.4f} 1/m")

fig = render_drape(f"{prefix}_hillshade.tif", f"{prefix}_direct_polyfit_laplacian.tif",
                   base_is_hillshade=True, transparency=30, cmap="RdBu",
                   vmin=-curv_lim, vmax=curv_lim,
                   colorbar=True, colorbar_label="Curvature (1/m)", colorbar_position="right",
                   style=style)
save_figure(fig, f"{prefix}_curvature_plot.png", style=style)
display(Image(f"{prefix}_curvature_plot.png", width=800))

## Step 3: A histogram of topographic gradients

In landscapes where hillslopes erode quickly, gradients pile up near a **threshold** value (often around 0.6 to 0.8 m/m), because slopes steeper than this fail by landsliding. A histogram with a peak at high gradients, or a part of the landscape where gradients stop increasing even though channels are steeper, can be a sign of rapid, transient erosion.

We read the gradient raster with `rasterio`, ignore the most extreme 0.5% of values at each end, and choose the bin width with the Freedman–Diaconis rule (bin width = 2 × interquartile range × n<sup>−1/3</sup>).

In [ ]:
with rasterio.open(f"{prefix}_direct_polyfit_slope.tif") as src:
    data = src.read(1, masked=True)
vals = data.compressed()
vals = vals[np.isfinite(vals)]

# --- (Optional) Convert units for readability ---
# vals = np.degrees(np.arctan(vals))   # m/m -> degrees
# vals = vals * 100                    # m/m -> %

# --- Clip the extreme tails (spikes and noise) ---
lo, hi = np.percentile(vals, [0.5, 99.5])
vals_clipped = vals[(vals >= lo) & (vals <= hi)]

# --- Freedman-Diaconis rule for the bin width ---
iqr = np.subtract(*np.percentile(vals_clipped, [75, 25]))
bin_width = 2 * iqr * (len(vals_clipped) ** (-1/3))
if bin_width <= 0:
    bins = 50
else:
    bins = max(10, int(np.ceil((vals_clipped.max() - vals_clipped.min()) / bin_width)))

print(f"Median gradient: {np.median(vals):.3f} m/m")
print(f"90th percentile gradient: {np.percentile(vals, 90):.3f} m/m")

plt.figure(figsize=(7.5, 4.5))
plt.hist(vals_clipped, bins=bins, color="#2a6fbb", edgecolor="white")
plt.xlabel("Topographic gradient (m/m)")
plt.ylabel("Pixel count")
plt.title("Histogram of topographic gradients (0.5–99.5th pct clipped)")
plt.tight_layout()
plt.savefig(f"{prefix}_slope_histogram.png", dpi=300)
plt.show()

## Step 4: Select basins and run a chi analysis

### What is chi?

A typical channel profile is a plot of elevation against flow distance. But it is hard to see changes in channel steepness on it, because headwater channels are steeper than downstream channels. The gradient of channels scales with drainage area, so to compare steepness you need to control for drainage area. The **stream power law** says that, in steady state, channel slope $S$ and drainage area $A$ are related by

$$S = k_{sn} A^{-\theta}, \qquad \theta = m/n .$$

* $\theta = m/n$ is the **concavity index**. We use the common default **m/n = 0.45**.
* $k_{sn}$ is the **normalised channel steepness index**. In many landscapes it is closely correlated with the erosion rate. See: *Kirby, E., Whipple, K., 2001. Quantifying differential rock-uplift rates via stream profile analysis. Geology 29, 415–418. https://doi.org/10.1130/0091-7613(2001)029<0415:QDRURV>2.0.CO;2*

A clever way to measure $k_{sn}$ is to integrate drainage area as you move upstream. This gives a transformed distance along the channel called **chi** ($\chi$):

$$\chi = \int_{0}^{x} \left(\frac{A_0}{A(x')}\right)^{m/n} dx' .$$

If you plot elevation against $\chi$, a channel in steady state is a straight line, and its **gradient is $k_{sn}$** (with $A_0 = 1$ m², the `lsdtt3` default). In a **transient** landscape the chi profile has kinks: a steep lower segment and a gentler upper segment, separated by a knickpoint. See: *Perron, J.T., Royden, L., 2013. An integral approach to bedrock river profile analysis. Earth Surface Processes and Landforms 38, 570–576. https://doi.org/10.1002/esp.3302*

### The basins

We give `lsdtt-chi-analysis` a CSV of outlet points with `latitude` and `longitude` columns. The basins are numbered from **0** in the order of the CSV rows.

**Important**: the DEM must contain the **entire** drainage basin. A basin cut by the edge of the DEM (or by nodata, such as the sea) is dropped, because its chi profile would be wrong. If you lose a basin, make the box in Step 1 bigger.

### Edit code block below for your site!

* Change the latitude and longitude of your basin outlets. Each outlet is `[latitude, longitude]` in square brackets, with a comma after each outlet apart from the last one, and all of them in an outer pair of square brackets:

```
[ [a, b],
  [c, d],
  [e, f] ]
```

In [ ]:
data = [ [37.94234239772101, 15.790076672680563],
         [37.93291942111813, 15.792125654538383],
         [38.06143176957989, 16.060604039819015],
         [38.094672987877786, 15.677112531901733],
         [38.20455853467961, 15.67642588617276],
         [38.12546843046537, 16.1230887795859] ]

outlets = pd.DataFrame(data, columns=["latitude", "longitude"])
outlets.to_csv(f"{prefix}_outlets.csv", index=False)
outlets

### The art of basin selection

The points we pick on a map are rarely exactly on the channel that the program computes, so we **snap** them: `basin_outlet_snapping_method: downstream` follows the flow path downstream from each point until it reaches a pixel with at least `basin_outlet_contributing_pixels_threshold` upslope pixels, i.e. a channel pixel. So pick each outlet on, or just upstream of, the river. The snapped outlet can be a little downstream of your point; Figure 4 shows both.

(This is simpler than in `lsdtt2`, which snapped to channel junctions and needed `search_radius_nodes`.)

### Run `lsdtt-chi-analysis`

The main parameters:

* `threshold_contributing_pixels: 500`: a channel starts where 500 pixels drain to a point. With 30 m pixels that is 500 × 900 m² = 0.45 km².
* `basin_outlet_fname`, `basin_outlet_snapping_method`, `basin_outlet_contributing_pixels_threshold`: the outlets and how to snap them (above).
* `n_largest_basins: 0`: keep all of our basins. (The default, 5, keeps only the 5 biggest *complete* basins.)
* `m_over_n: 0.45`: the concavity index.
* `write_chi_data_maps: true` with `chi_data_maps_write_vector_format: csv`: write every channel pixel with its chi, elevation, drainage area and flow distance (`calabria_chi_data_maps.csv`), the snapped outlets (`calabria_chi_basin_outlets.csv`) and the basin outlines (`calabria_chi_basin_polygons.fgb`).
* `fit_chi_networks: true`: fit each chi profile with straight-line **segments**, split wherever the data move more than `fit_critical_divergence_metres` (here 10 m) from a straight line. Each segment's gradient is its $k_{sn}$. Outputs: `calabria_chi_fit_nodes_mn_0p450.csv` and `calabria_chi_fit_segments_mn_0p450.csv` (`mn_0p450` means m/n = 0.450).
* `write_chi_fit_segment_lines: true`: the same segments as map lines, `calabria_chi_fit_segment_lines_mn_0p450.fgb`, so we can map $k_{sn}$ in Step 7.

In the printed output, look for the line **`Complete basins: N / 6`**. If N is less than 6, some basins touch the edge of the DEM or the sea.

In [ ]:
threshold = 500

with open(f"{prefix}_chi.param", "w") as f:
    f.write(f"read_fname: {prefix}_DEM.tif\n")
    f.write(f"write_fname: {prefix}\n")
    f.write(f"threshold_contributing_pixels: {threshold}\n")
    f.write(f"basin_outlet_fname: {prefix}_outlets.csv\n")
    f.write("basin_outlet_snapping_method: downstream\n")
    f.write(f"basin_outlet_contributing_pixels_threshold: {threshold}\n")
    f.write("n_largest_basins: 0\n")
    f.write("m_over_n: 0.45\n")
    f.write("write_chi_data_maps: true\n")
    f.write("chi_data_maps_write_vector_format: csv\n")
    f.write("fit_chi_networks: true\n")
    f.write("fit_critical_divergence_metres: 10\n")
    f.write("write_fit_tables: true\n")
    f.write("write_chi_fit_segment_lines: true\n")

!lsdtt-chi-analysis ./ {prefix}_chi.param

In [ ]:
!ls -lh {prefix}_chi*

### Figure 4: Basin map

We load the basin outlines, check whether any of them touches the edge of the DEM, and see how far each outlet point was moved by snapping. Then we map them: the basins semi-transparent and numbered, the points we asked for as red crosses and the snapped outlets as black circles.

In [ ]:
from shapely.geometry import box

basins = load_vector(f"{prefix}_chi_basin_polygons.fgb")
basins["area_km2"] = (basins.geometry.area / 1e6).round(2)

# Does any basin touch the edge of the DEM? (It should not.)
with rasterio.open(f"{prefix}_DEM.tif") as src:
    dem_edge = box(*src.bounds).exterior.buffer(45)   # 1.5 pixels wide
basins["touches_dem_edge"] = basins.geometry.intersects(dem_edge)

requested = gpd.GeoDataFrame(outlets, geometry=gpd.points_from_xy(outlets.longitude, outlets.latitude),
                             crs="EPSG:4326")
snapped = pd.read_csv(f"{prefix}_chi_basin_outlets.csv")
snapped = gpd.GeoDataFrame(snapped, geometry=gpd.points_from_xy(snapped.longitude, snapped.latitude),
                           crs="EPSG:4326")

basins.drop(columns="geometry")

In [ ]:
fig = render_hillshade(f"{prefix}_hillshade.tif", shade=False, style=hs_style)
basins["label"] = basins["basin_key"].astype(int)
fig = render_basins(basins, label_col="label", fill="orange", transparency=50,
                    style=replace(style, line_width="0.8p", line_color="black"), fig=fig)
fig = render_points(requested, symbol="x", pen="1.5p,red",
                    style=replace(style, point_size="0.3c"), fig=fig)
fig = render_points(snapped, symbol="c", pen="0.5p,black",
                    style=replace(style, point_size="0.2c"), fig=fig)
save_figure(fig, f"{prefix}_basins_plot.png", style=style)
display(Image(f"{prefix}_basins_plot.png", width=800))

## Step 5: Chi profiles and channel long profiles

First we load the channel data. Every channel pixel has a `basin_key`, `chi`, `elevation`, `flow_distance` (distance upstream from the basin outlet, in m) and `drainage_area`.

In [ ]:
chi_points = pd.read_csv(f"{prefix}_chi_data_maps.csv")
print("Basins:", sorted(chi_points["basin_key"].unique()))
chi_points.head()

### Edit code block below for your site!

* Change the basin number to look at different basins.

In [ ]:
basin = 2

### Figure 5: Chi–elevation plot

Each point is a channel pixel, coloured by its flow distance. Look for **kinks**: a change in gradient means a change in $k_{sn}$. If the tributaries (the short branches) plot on top of the main stem, m/n = 0.45 is about right.

In [ ]:
df_single_basin = chi_points[chi_points["basin_key"] == basin]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df_single_basin.chi, df_single_basin.elevation,
                c=df_single_basin.flow_distance / 1000, s=3, cmap="viridis")
fig.colorbar(sc, ax=ax, label="Flow distance (km)")
ax.set_xlabel(r"$\chi$ (m)")
ax.set_ylabel("Elevation (m)")
ax.text(0.05, 0.92, f"Basin {basin}", transform=ax.transAxes)

chi_profile_name = f"{prefix}_chi_profile_basin{basin}.png"
fig.savefig(chi_profile_name, bbox_inches="tight", dpi=300)
plt.show()

### Figure 6: Channel long profile

The traditional view, elevation against flow distance, with the points coloured by chi. Compare the two plots: the knickpoints are often much easier to see in the chi plot.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df_single_basin.flow_distance / 1000, df_single_basin.elevation,
                c=df_single_basin.chi, s=3, cmap="viridis")
fig.colorbar(sc, ax=ax, label=r"$\chi$ (m)")
ax.set_xlabel("Flow distance (km)")
ax.set_ylabel("Elevation (m)")
ax.text(0.05, 0.92, f"Basin {basin}", transform=ax.transAxes)

channel_profile_name = f"{prefix}_channel_profile_basin{basin}.png"
fig.savefig(channel_profile_name, bbox_inches="tight", dpi=300)
plt.show()

## Step 6: A swath profile

You could collect elevations along a transect to see trends in elevation, for example across a river valley to see what the hillslopes are doing. But topography is noisy, with ridges and gullies, so a single transect can mislead. A **swath profile** takes a baseline (the middle of your transect) and, at each distance along it, summarises all the elevations in a strip on either side. The median along the swath looks much like a single transect, and the spread shows how much the elevations vary. The method is from:

*Hergarten, S., Robl, J., Stüwe, K., 2014. Extracting topographic swath profiles across curved geomorphic features. Earth Surface Dynamics 2, 97–104. https://doi.org/10.5194/esurf-2-97-2014*

`lsdtt-swath-profile` reads the baseline as a **line** in a vector file, in the DEM's coordinate system (metres). We make the line from two latitude–longitude points with `geopandas`, reproject it to the DEM's coordinate system, and save it as `calabria_swath_baseline.fgb`.

### Edit code block below for your site!

* Update the two `[latitude, longitude]` pairs to set the ends of your swath.
* `half_width_metres` is how far either side of the line pixels are included.

In [ ]:
swath_ends = [ [38.071180814769235, 15.821177352750635],
               [38.10306680861067, 15.796114793399402] ]
half_width_metres = 1000

In [ ]:
from shapely.geometry import LineString

with rasterio.open(f"{prefix}_DEM.tif") as src:
    dem_crs = src.crs

baseline = gpd.GeoDataFrame(
    {"name": ["swath"]},
    geometry=[LineString([(lon, lat) for lat, lon in swath_ends])],
    crs="EPSG:4326",
).to_crs(dem_crs)
baseline.to_file(f"{prefix}_swath_baseline.fgb")
print(f"Baseline length: {baseline.geometry.length.iloc[0]:.0f} m")

Now run `lsdtt-swath-profile`:

* `baseline_vector_fname`: the line we just wrote.
* `half_width_metres`: only pixels within this distance of the line are used.
* `bin_width_metres: 90`: the elevations are summarised in 90 m bins along the line.

Outputs: `calabria_swath_profile.csv` (for each bin: the distance along the line, the number of pixels, the mean and the 0th, 5th, ..., 100th percentiles of elevation) and `calabria_swath_along_axis.tif` (the distance along the line of every pixel in the swath, which shows where the swath is).

In [ ]:
with open(f"{prefix}_swath.param", "w") as f:
    f.write(f"read_fname: {prefix}_DEM.tif\n")
    f.write(f"write_fname: {prefix}\n")
    f.write(f"baseline_vector_fname: {prefix}_swath_baseline.fgb\n")
    f.write(f"half_width_metres: {half_width_metres}\n")
    f.write("bin_width_metres: 90\n")
    f.write("write_distance_rasters: true\n")

!lsdtt-swath-profile ./ {prefix}_swath.param

### Figure 7: The swath footprint

This shows where the swath data come from: each pixel in the swath is coloured by its distance along the baseline (black line).

In [ ]:
fig = render_drape(f"{prefix}_hillshade.tif", f"{prefix}_swath_along_axis.tif",
                   base_is_hillshade=True, transparency=40, cmap="jet",
                   colorbar=True, colorbar_label="Distance along swath (m)",
                   colorbar_position="right", style=style)
fig = render_channels(baseline, style=replace(style, line_width="1.5p", line_color="black"), fig=fig)
save_figure(fig, f"{prefix}_swath_footprint_plot.png", style=style)
display(Image(f"{prefix}_swath_footprint_plot.png", width=800))

### Figure 8: The swath profile

`plot_swath` from `lsdviztools3` draws the minimum-to-maximum envelope (lightest), the interquartile range (25th to 75th percentile), the median (solid line) and the mean (dashed line).

In [ ]:
from lsdviztools3.io.swath import load_swath
from lsdviztools3.charts import plot_swath

swath = load_swath(f"{prefix}_swath_profile.csv")
fig, ax = plot_swath(swath, bands=((0, 100), (25, 75)), distance_units="km")
fig.set_size_inches(10, 5)
fig.tight_layout()
fig.savefig(f"{prefix}_swath_plot.png", dpi=300)
plt.show()

## Step 7: Channel steepness

### Figure 9: Stacked chi profiles

`load_chi_fit` reads the fit nodes and segments and gives every channel pixel the $k_{sn}$ of its fitted segment. `plot_chi_stacked` puts all the basins side by side (each shifted along the chi axis), coloured by $\log_{10} k_{sn}$ on one shared colour scale, so their steepness can be compared. In a transient landscape you will see the colour change from steep (yellow) low down to gentle (purple) higher up the profiles.

In [ ]:
from lsdviztools3.io.chi import load_chi_fit
from lsdviztools3.charts import plot_chi_profile, plot_chi_stacked

chi_fit = load_chi_fit(f"{prefix}_chi_fit_nodes_mn_0p450.csv",
                       f"{prefix}_chi_fit_segments_mn_0p450.csv")

fig, ax = plot_chi_stacked(chi_fit, cmap="viridis")
fig.tight_layout()
fig.savefig(f"{prefix}_chi_stacked.png", dpi=300)
plt.show()

`plot_chi_profile` draws a single basin, with the fitted straight-line segments on top of the data. Here is the basin you chose in Step 5.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_chi_profile(chi_fit, basin=basin, ax=ax, cmap="viridis", point_size=3)
fig.tight_layout()
fig.savefig(f"{prefix}_chi_fit_basin{basin}.png", dpi=300)
plt.show()

`lsdtt-chi-analysis` also writes a summary of $k_{sn}$ for each basin. The `main_stem_*` columns use only the main stem. $k_{sn}$ here has units of m$^{2\theta}$ = m$^{0.9}$.

In [ ]:
ksn_summary = pd.read_csv(f"{prefix}_chi_ksn_basin_summary_mn_0p450.csv")
ksn_summary[["basin_index", "n_segments", "median_k_sn", "chi_weighted_mean_k_sn",
             "main_stem_median_k_sn"]].round(1)

### Figure 10: A map of $k_{sn}$

The channels in our basins coloured by the $k_{sn}$ of their fitted chi segment (`color_by="k_sn"`, on a log scale), with the basin outlines in black. Look for where along the rivers $k_{sn}$ changes: in a transient landscape the knickpoints often sit at similar elevations, or at similar distances from the coast, in neighbouring basins. We leave out segments with $k_{sn} \le 0$ (flat or slightly uphill fits), which cannot be shown on a log scale.

In [ ]:
seg_lines = load_vector(f"{prefix}_chi_fit_segment_lines_mn_0p450.fgb")

outlines = basins.copy()
outlines["geometry"] = basins.boundary

fig = render_hillshade(f"{prefix}_hillshade.tif", shade=False, style=hs_style)
fig = render_channels(outlines, style=replace(style, line_width="0.8p", line_color="black"), fig=fig)
fig = render_channels(seg_lines[seg_lines["k_sn"] > 0], color_by="k_sn", log_color=True,
                      cmap="viridis", colorbar=True, colorbar_label="log@-10@- k@-sn@-",
                      style=replace(style, line_width="1.5p"), fig=fig)
save_figure(fig, f"{prefix}_ksn_plot.png", style=style)
display(Image(f"{prefix}_ksn_plot.png", width=800))

## Things to try

* **Your own site.** Change the box in Step 1, the outlets in Step 4 and the swath in Step 6.
* **Concavity.** Change `m_over_n` to 0.3 or 0.6 and rerun Steps 4, 5 and 7 (the output names change too: `mn_0p300`, `mn_0p600`). For which value do the tributaries line up best with the main stem?
* **Segment fitting.** Set `fit_critical_divergence_metres` to 5 or 30. How does the number of segments change?
* **Smoothing.** Change `polyfit_lambda_metres` to 150 in Step 2. How do the gradient and curvature maps change?
* **A swath of something else.** Add `value_raster_fname: calabria_direct_polyfit_slope.tif` to the swath parameter file to get a swath of gradient instead of elevation (change `value_label` in `plot_swath` to match). Do steeper channels (Figure 10) line up with steeper hillslopes?

## Citing lsdtt3

You can find the citation for `lsdtt3` by running any of its programs with `-c`, for example `!lsdtt-chi-analysis -c`.